<a href="https://colab.research.google.com/github/SIRLLON/AT-estdados/blob/main/at.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AT: Algoritmos e Estruturas de Dados em Python



## Exercício 1: Heap Binária do Zero e Manutenção de Top-K

### Detalhes Teóricos e Impessoais
- **Propriedade da Heap**: Uma min-heap binária é uma árvore binária quase completa na qual, para qualquer nó $i$, o valor de $i$ é maior ou igual ao valor de seu pai. Isso garante que a raiz sempre contenha o menor elemento.
- **Estratégia para Top-K**: Para obter os $K$ maiores elementos de um fluxo contínuo (tamanho $N$, potencialmente infinito), mantém-se uma min-heap de tamanho fixo $K$. Cada novo elemento do fluxo é comparado com o menor dos elementos atualmente selecionados (que fica na raiz da heap). Se o novo elemento for maior que a raiz, a raiz é removida (`pop`) e o novo elemento é inserido (`push`). Caso contrário, o elemento é descartado. Ao final, a heap conterá exatamente os $K$ maiores elementos do fluxo.
- **Justificativa de Custo e Memória**:
  - **Tempo**: $O(N \log K)$ comparado com $O(N \log N)$ de ordenar ou inserir tudo. Cada uma das $N$ operações de inserção e exclusão na heap de tamanho $K$ custa no máximo $O(\log K)$ comparações e swaps.
  - **Memória**: $O(K)$ espaço adicional para armazenar a heap de tamanho fixo, em contraste com $O(N)$ necessário para armazenar todo o fluxo.
  - **Escolha da Heap**: Utiliza-se uma **min-heap** porque ela disponibiliza em sua raiz o menor valor do conjunto de candidatos ao Top-K em tempo $O(1)$, facilitando o descarte imediato de elementos menores.

In [1]:
# Exercício 1: Min-Heap binária
class BinaryHeap:
    def __init__(self):
        self.heap = []
        self.comparisons = 0
        self.swaps = 0

    def _parent(self, i):
        return (i - 1) // 2

    def _left(self, i):
        return 2 * i + 1

    def _right(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        # Registra troca efetuada
        self.swaps += 1
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    def __len__(self):
        return len(self.heap)

    def peek(self):
        if not self.heap:
            return None
        return self.heap[0]

    def push(self, x):
        self.heap.append(x)
        self._sift_up(len(self.heap) - 1)

    def pop(self):
        if not self.heap:
            return None
        if len(self.heap) == 1:
            return self.heap.pop()
        root = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._sift_down(0)
        return root

    def _sift_up(self, i):
        while i > 0:
            parent = self._parent(i)
            # Incrementa contador comparações
            self.comparisons += 1
            if self.heap[i] < self.heap[parent]:
                self._swap(i, parent)
                i = parent
            else:
                break

    def _sift_down(self, i):
        n = len(self.heap)
        while self._left(i) < n:
            left = self._left(i)
            right = self._right(i)
            smallest = left
            # Incrementa contador comparações
            self.comparisons += 1
            if right < n:
                self.comparisons += 1
                if self.heap[right] < self.heap[left]:
                    smallest = right
            self.comparisons += 1
            if self.heap[smallest] < self.heap[i]:
                self._swap(i, smallest)
                i = smallest
            else:
                break

def manter_top_k(fluxo, k):
    # Cria heap vazia
    heap = BinaryHeap()
    for item in fluxo:
        if len(heap) < k:
            # Insere elemento inicial
            heap.push(item)
        else:
            # Compara com menor atual
            heap.comparisons += 1
            if item > heap.peek():
                # Remove menor elemento
                heap.pop()
                # Insere novo elemento
                heap.push(item)
    return heap

# Exercício 1: Testes
fluxo_teste = [15, 3, 22, 10, 8, 45, 1, 99, 14, 30]
k_teste = 4
heap_top_k = manter_top_k(fluxo_teste, k_teste)
print("Fluxo original:", fluxo_teste)
print("Top-K maiores elementos:", sorted(heap_top_k.heap, reverse=True))
print(f"Total de comparações: {heap_top_k.comparisons}")
print(f"Total de swaps: {heap_top_k.swaps}")

Fluxo original: [15, 3, 22, 10, 8, 45, 1, 99, 14, 30]
Top-K maiores elementos: [99, 45, 30, 22]
Total de comparações: 28
Total de swaps: 9


## Exercício 2: Heap como Array: Validação Estrutural e Build-Heap Eficiente

### Detalhes Teóricos e Impessoais
- **Validação de Heap (`is_valid_heap`)**: Verifica-se se um array representa uma min-heap válida. Para todo índice $i$, se existir um filho esquerdo (índice $2i + 1$) ou direito (índice $2i + 2$), os valores destes filhos devem ser maiores ou iguais ao valor em $i$.
- **Construção de Heap Bottom-Up (`build_heap`)**: O algoritmo de Floyd percorre os nós a partir do último nó que possui filhos (índice $\lfloor N/2 \rfloor - 1$) até o nó raiz (índice 0) aplicando a operação de `sift_down` em cada etapa.
- **Justificativa de Complexidade**:
  - **Tempo (`build_heap`)**: $O(N)$ no pior caso. A soma das alturas de todos os nós é $\sum_{h=0}^{\log N} \frac{N}{2^{h+1}} O(h) = O(N)$.
  - **Tempo (Inserção Incremental)**: $O(N \log N)$ no pior caso, pois cada elemento inserido pode subir até a altura total da árvore.
  - **Comparação**: O ganho de desempenho da estratégia Bottom-Up torna-se evidente para $N$ grande, já que a maioria dos nós está nos níveis inferiores e realiza poucas ou nenhuma comparação durante a construção.

In [2]:
# Exercício 2: Métodos de array
def is_valid_heap(arr):
    n = len(arr)
    # Cobre casos pequenos
    if n <= 1:
        return True
    for i in range(n):
        left = 2 * i + 1
        right = 2 * i + 2
        if left < n:
            if arr[left] < arr[i]:
                return False
        if right < n:
            if arr[right] < arr[i]:
                return False
    return True

def build_heap(arr):
    n = len(arr)
    heap_arr = list(arr)
    comparisons = 0
    swaps = 0

    def sift_down_built(i):
        nonlocal comparisons, swaps
        while 2 * i + 1 < n:
            left = 2 * i + 1
            right = 2 * i + 2
            smallest = left
            comparisons += 1
            if right < n:
                comparisons += 1
                if heap_arr[right] < heap_arr[left]:
                    smallest = right
            comparisons += 1
            if heap_arr[smallest] < heap_arr[i]:
                swaps += 1
                heap_arr[i], heap_arr[smallest] = heap_arr[smallest], heap_arr[i]
                i = smallest
            else:
                break

    # Começa no último pai
    for j in range(n // 2 - 1, -1, -1):
        sift_down_built(j)
    return heap_arr, comparisons, swaps

def build_heap_incremental(arr):
    heap = BinaryHeap()
    for item in arr:
        heap.push(item)
    return heap.heap, heap.comparisons, heap.swaps

# Exercício 2: Testes
valid_arr = [2, 5, 8, 12, 10, 15]
invalid_arr = [15, 12, 8, 5, 10, 2]
print(f"Array {valid_arr} é válido? {is_valid_heap(valid_arr)}")
print(f"Array {invalid_arr} é válido? {is_valid_heap(invalid_arr)}")

# Comparação empírica de eficiência
tamanhos = [10, 100, 1000]
print("\nComparativo de Construção de Heap:")
for size in tamanhos:
    import random
    random.seed(42)
    dados = [random.randint(1, 10000) for _ in range(size)]
    _, comp_b, swap_b = build_heap(dados)
    _, comp_i, swap_i = build_heap_incremental(dados)
    print(f"Tamanho {size}:")
    print(f"  Bottom-up (Floyd): {comp_b} comparações, {swap_b} swaps")
    print(f"  Incremental:      {comp_i} comparações, {swap_i} swaps")

Array [2, 5, 8, 12, 10, 15] é válido? True
Array [15, 12, 8, 5, 10, 2] é válido? False

Comparativo de Construção de Heap:
Tamanho 10:
  Bottom-up (Floyd): 20 comparações, 4 swaps
  Incremental:      13 comparações, 5 swaps
Tamanho 100:
  Bottom-up (Floyd): 253 comparações, 63 swaps
  Incremental:      196 comparações, 99 swaps
Tamanho 1000:
  Bottom-up (Floyd): 2752 comparações, 687 swaps
  Incremental:      2142 comparações, 1148 swaps


## Exercício 3: Heap e Escolha de Estratégia: Deleção Arbitrária e Trade-offs

### Detalhes Teóricos e Impessoais
- **Operações Arbitrárias**: `contains(x)` localiza o elemento na representação interna e `delete(x)` remove uma ocorrência de $x$.
- **Regra de Desempate e Duplicatas**: Remove-se a ocorrência de menor índice no array interno. Para preservar a estrutura da heap, o elemento no menor índice encontrado é substituído pelo último elemento da heap, reduzindo o tamanho do array por 1. Em seguida, aplica-se `sift_down` ou `sift_up` a partir do índice modificado para restaurar a invariante da min-heap.
- **Justificativa de Otimização e Trade-offs**:
  - **Heap Sem Otimização**: A busca por $x$ leva tempo $O(N)$ e a remoção e reorganização leva $O(\log N)$. O custo total é $O(N)$ em tempo e $O(1)$ em memória adicional.
  - **Heap Com Otimização (Dicionário de Índices)**: Mantém-se um dicionário que mapeia cada valor para um conjunto de índices onde ele ocorre. O mapa é atualizado a cada `_swap`, `push` e `pop` em tempo $O(1)$.
  - **Trade-off**: A busca pelo índice de $x$ passa a ser de custo médio $O(1)$ amortizado. O custo de deleção cai para $O(\log N)$. Contudo, a complexidade de manutenção do código aumenta consideravelmente e o consumo de memória cresce para $O(N)$ espaço extra.

In [3]:
# Exercício 3: Heap otimizada
class OptimizedBinaryHeap:
    def __init__(self):
        self.heap = []
        self.index_map = {}  # valor -> set de índices
        self.comparisons = 0
        self.swaps = 0

    def _parent(self, i):
        return (i - 1) // 2

    def _left(self, i):
        return 2 * i + 1

    def _right(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self.swaps += 1
        val_i = self.heap[i]
        val_j = self.heap[j]
        # Atualiza posições no mapa
        self.index_map[val_i].remove(i)
        self.index_map[val_i].add(j)
        self.index_map[val_j].remove(j)
        self.index_map[val_j].add(i)
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    def __len__(self):
        return len(self.heap)

    def peek(self):
        if not self.heap:
            return None
        return self.heap[0]

    def push(self, x):
        # Registra índice no mapa
        idx = len(self.heap)
        self.heap.append(x)
        if x not in self.index_map:
            self.index_map[x] = set()
        self.index_map[x].add(idx)
        self._sift_up(idx)

    def pop(self):
        if not self.heap:
            return None
        if len(self.heap) == 1:
            val = self.heap.pop()
            self.index_map[val].remove(0)
            if not self.index_map[val]:
                del self.index_map[val]
            return val
        root = self.heap[0]
        # Remove elemento raiz
        last_val = self.heap.pop()
        self.index_map[root].remove(0)
        if not self.index_map[root]:
            del self.index_map[root]

        if self.heap:
            self.index_map[last_val].remove(len(self.heap))
            self.heap[0] = last_val
            self.index_map[last_val].add(0)
            self._sift_down(0)
        return root

    def contains(self, x):
        # Consulta rápida no mapa
        return x in self.index_map and len(self.index_map[x]) > 0

    def delete(self, x):
        if not self.contains(x):
            return False
        # Obtém menor índice associado
        idx = min(self.index_map[x])
        n = len(self.heap)

        # Trata último elemento
        if idx == n - 1:
            self.heap.pop()
            self.index_map[x].remove(idx)
            if not self.index_map[x]:
                del self.index_map[x]
            return True

        last_val = self.heap.pop()
        self.index_map[x].remove(idx)
        if not self.index_map[x]:
            del self.index_map[x]

        self.index_map[last_val].remove(n - 1)
        self.heap[idx] = last_val
        self.index_map[last_val].add(idx)

        # Reposiciona o elemento substituído
        self.comparisons += 1
        parent = self._parent(idx)
        if idx > 0 and self.heap[idx] < self.heap[parent]:
            self._sift_up(idx)
        else:
            self._sift_down(idx)
        return True

    def _sift_up(self, i):
        while i > 0:
            parent = self._parent(i)
            self.comparisons += 1
            if self.heap[i] < self.heap[parent]:
                self._swap(i, parent)
                i = parent
            else:
                break

    def _sift_down(self, i):
        n = len(self.heap)
        while self._left(i) < n:
            left = self._left(i)
            right = self._right(i)
            smallest = left
            self.comparisons += 1
            if right < n:
                self.comparisons += 1
                if self.heap[right] < self.heap[left]:
                    smallest = right
            self.comparisons += 1
            if self.heap[smallest] < self.heap[i]:
                self._swap(i, smallest)
                i = smallest
            else:
                break

# Exercício 3: Testes
heap_op = OptimizedBinaryHeap()
for val in [10, 20, 10, 30, 10, 40]:
    heap_op.push(val)

print("Heap antes do delete:", heap_op.heap)
print("Contém 20?", heap_op.contains(20))
print("Deletando um 10:", heap_op.delete(10))
print("Heap após deletar 10:", heap_op.heap)
print("Deletando 20:", heap_op.delete(20))
print("Heap após deletar 20:", heap_op.heap)

Heap antes do delete: [10, 10, 10, 30, 20, 40]
Contém 20? True
Deletando um 10: True
Heap após deletar 10: [10, 20, 10, 30, 40]
Deletando 20: True
Heap após deletar 20: [10, 30, 10, 40]


## Exercício 4: Trie do Zero: Inserção, Fim de Palavra e Listagem Lexicográfica

### Detalhes Teóricos e Impessoais
- **TrieNode e Trie**: A árvore Trie armazena prefixos de palavras. Cada nó `TrieNode` possui um dicionário `children` mapeando caracteres para nós filhos e um booleano `is_end` indicando se aquele caractere finaliza uma palavra.
- **Listagem Lexicográfica (`list_words`)**: Para coletar todas as palavras armazenadas em ordem alfabética estável, executa-se uma travessia por busca em profundidade (DFS) a partir do nó raiz. Os filhos de cada nó são percorridos em ordem lexicográfica.
- **Justificativa de Custo e Memória**:
  - **Tempo (`insert`/`contains`)**: $O(L)$, onde $L$ é o comprimento da palavra pesquisada.
  - **Tempo (`list_words`)**: $O(W \times L)$ no pior caso, onde $W$ é o número de palavras e $L$ é o comprimento máximo de uma palavra, visitando cada nó exatamente uma vez.
  - **Memória**: $O(\Sigma \times N_{nodes})$ no pior caso, onde $\Sigma$ é o tamanho do alfabeto e $N_{nodes}$ é a quantidade de nós gerados na Trie.
  - **Poda Heurística**: Adiciona-se uma redução de custo controlada por parâmetro `max_depth` ou `max_words`. Essa poda é heurística porque interrompe a travessia precocemente sem garantia de coletar todas as chaves existentes se as restrições forem violadas.

In [4]:
# Exercício 4: Classe Trie
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end = True

    def contains(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                return False
            node = node.children[char]
        return node.is_end

    def list_words(self, max_depth=None, max_words=None):
        palavras = []
        poda_aplicada = False

        def dfs(node, path, depth):
            nonlocal poda_aplicada
            if max_words is not None and len(palavras) >= max_words:
                poda_aplicada = True
                return
            if max_depth is not None and depth > max_depth:
                poda_aplicada = True
                return
            if node.is_end:
                palavras.append(path)
            # Ordena chaves alfabeticamente
            for char in sorted(node.children.keys()):
                dfs(node.children[char], path + char, depth + 1)

        dfs(self.root, "", 0)
        if poda_aplicada:
            print("LOG: Poda aplicada na busca.")
        return palavras

# Exercício 4: Testes
trie = Trie()
for w in ["bola", "bolo", "bolacha", "casa", "carro"]:
    trie.insert(w)

print("Palavras na Trie (lexicográfica):", trie.list_words())
print("Contém 'bolo'?", trie.contains("bolo"))
print("Contém 'bol' (prefixo)?", trie.contains("bol"))
print("Teste Poda (max_depth=4):", trie.list_words(max_depth=4))

Palavras na Trie (lexicográfica): ['bola', 'bolacha', 'bolo', 'carro', 'casa']
Contém 'bolo'? True
Contém 'bol' (prefixo)? False
LOG: Poda aplicada na busca.
Teste Poda (max_depth=4): ['bola', 'bolo', 'casa']


## Exercício 5: Autocomplete com Trie: Limite K, Ordenação e Determinismo

### Detalhes Teóricos e Impessoais
- **Autocomplete Determinístico (`autocomplete`)**:
  - **Fase 1**: Localiza-se o nó final correspondente ao prefixo informado (complexidade $O(|P|)$, onde $|P|$ é o tamanho do prefixo).
  - **Fase 2**: A partir deste nó, realiza-se uma travessia DFS na subárvore coletando palavras completas. Os caracteres vizinhos em cada nó são explorados em ordem lexicográfica ordenada para garantir determinismo.
- **Priorização Gulosa**: Como item adicional de otimização, priorizam-se palavras mais curtas como critério de melhor qualidade. Em caso de empate no comprimento, o desempate é feito de forma lexicográfica (ordem alfabética).
- **Justificativa de Complexidade**:
  - **Tempo**: A localização do nó leva $O(|P|)$. A exploração da subárvore para coletar as palavras candidatas leva no pior caso $O(N_{sub} \times L)$, onde $N_{sub}$ é a quantidade de nós sob o prefixo. A ordenação do Top-$K$ das sugestões geradas leva $O(M \log M)$, onde $M$ é o total de sugestões da subárvore obtida.

In [5]:
# Exercício 5: Autocomplete
def starts_with(self, prefix):
    node = self.root
    for char in prefix:
        if char not in node.children:
            return False
        node = node.children[char]
    return True

Trie.starts_with = starts_with

def find_node(self, prefix):
    # Encontra nó do prefixo
    node = self.root
    for char in prefix:
        if char not in node.children:
            return None
        node = node.children[char]
    return node

Trie.find_node = find_node

def autocomplete(self, prefix, k):
    node = self.find_node(prefix)
    if not node:
        return []

    sugestoes = []

    def dfs(curr_node, current_path):
        if curr_node.is_end:
            sugestoes.append(current_path)
        for char in sorted(curr_node.children.keys()):
            dfs(curr_node.children[char], current_path + char)

    dfs(node, prefix)

    # Ordena: comprimento e alfabético
    sugestoes.sort(key=lambda s: (len(s), s))
    return sugestoes[:k]

Trie.autocomplete = autocomplete

# Exercício 5: Testes
trie_ac = Trie()
for w in ["car", "cart", "carro", "caravana", "casa", "cara"]:
    trie_ac.insert(w)

print("Autocomplete para 'car' (k=3):", trie_ac.autocomplete("car", 3))
print("Autocomplete para 'casa' (k=3):", trie_ac.autocomplete("casa", 3))
print("Autocomplete para 'xyz' (k=3):", trie_ac.autocomplete("xyz", 3))

Autocomplete para 'car' (k=3): ['car', 'cara', 'cart']
Autocomplete para 'casa' (k=3): ['casa']
Autocomplete para 'xyz' (k=3): []


## Exercício 6: Autocorrect em Trie: Maior Prefixo e Heurística de Erro Limitado

### Detalhes Teóricos e Impessoais
- **Autocorrect com Maior Prefixo**: Se o termo digitado `query` existir na Trie, ele é retornado imediatamente. Caso não exista, busca-se a palavra que compartilha o maior prefixo comum.
- **Erro de Substituição e Poda**: Estende-se a busca para aceitar até 1 caractere de erro (substituição). Para evitar a explosão combinatória, implementa-se um limite de candidatos explorados `max_candidates`. Quando o limite é alcançado, a busca é interrompida.
- **Justificativa de Custo e Heurística**:
  - A busca exaustiva por erros em árvores Trie pode requerer a exploração de múltiplos ramos da árvore. Ao limitar a busca de substituição a um nível de erro ($E=1$) e implementar um teto de candidatos explorados, limita-se a complexidade a um custo previsível. O desempate entre palavras candidatas é resolvido de forma determinística: escolhe-se a palavra lexicograficamente menor em caso de igualdade de distância e tamanho de prefixo.

In [6]:
# Exercício 6: Autocorrect
def autocorrect(self, query, max_candidates=10):
    if self.contains(query):
        return query

    # Encontra maior prefixo compartilhado
    node = self.root
    prefix = ""
    for char in query:
        if char in node.children:
            prefix += char
            node = node.children[char]
        else:
            break

    candidatos = []
    poda_aplicada = False

    def buscar_com_erro(curr_node, word_so_far, idx, erros):
        nonlocal poda_aplicada
        if len(candidatos) >= max_candidates:
            poda_aplicada = True
            return

        if idx == len(query):
            if curr_node.is_end and erros <= 1:
                candidatos.append((erros, word_so_far))
            return

        char_alvo = query[idx]
        # Testa correspondência exata
        if char_alvo in curr_node.children:
            buscar_com_erro(curr_node.children[char_alvo], word_so_far + char_alvo, idx + 1, erros)

        # Testa substituição com erro
        if erros < 1:
            for char_sub in curr_node.children:
                if char_sub != char_alvo:
                    buscar_com_erro(curr_node.children[char_sub], word_so_far + char_sub, idx + 1, erros + 1)

    buscar_com_erro(self.root, "", 0, 0)

    if poda_aplicada:
        print("LOG: Poda de candidatos aplicada.")

    if not candidatos:
        # Retorna maior prefixo comum
        if prefix:
            sugs = []
            def coletar(n, path):
                if n.is_end:
                    sugs.append(path)
                for c in sorted(n.children.keys()):
                    coletar(n.children[c], path + c)
            coletar(self.find_node(prefix), prefix)
            return sorted(sugs)[0] if sugs else None
        return None

    # Ordena: erros, comprimento, lexicográfica
    candidatos.sort(key=lambda x: (x[0], len(x[1]), x[1]))
    return candidatos[0][1]

Trie.autocorrect = autocorrect

# Exercício 6: Testes
trie_ac = Trie()
for word in ["casa", "carro", "carroça", "caso", "pato"]:
    trie_ac.insert(word)

print("Corrigir 'caza' (substituição):", trie_ac.autocorrect("caza"))
print("Corrigir 'carr' (prefixo comum):", trie_ac.autocorrect("carr"))
print("Corrigir 'pat' (prefixo comum):", trie_ac.autocorrect("pat"))

Corrigir 'caza' (substituição): casa
Corrigir 'carr' (prefixo comum): carro
Corrigir 'pat' (prefixo comum): pato


## Exercício 7: Grafo Não Ponderado: BFS para Menor Caminho e Reconstrução

### Detalhes Teóricos e Impessoais
- **BFS e Reconstrução**: O algoritmo de busca em largura (BFS) explora vizinhanças nível a nível a partir de uma origem. Em grafos não ponderados, a primeira vez que um vértice é expandido garante-se o menor caminho até ele em quantidade de arestas. Reconstrói-se a rota salvando-se os caminhos completos ou os predecessores.
- **Heurística de Obrigatórios**: Para passar por um conjunto de nós obrigatórios, a heurística gulosa calcula a distância atual até cada obrigatório não visitado e escolhe como próximo destino o obrigatório mais próximo pelo menor caminho.
- **Justificativa de Complexidade**:
  - **Tempo (BFS)**: $O(V + E)$, onde $V$ representa a quantidade de vértices e $E$ o número de arestas visitados.
  - **Heurística**: A dificuldade cresce com o tamanho do conjunto de obrigatórios $M$. Encontrar a sequência ótima de visitas é equivalente ao Problema do Caixeiro Viajante (TSP), que é NP-difícil. A aproximação gulosa realiza no máximo $M$ buscas BFS, reduzindo a complexidade temporal para $O(M \times (V + E))$.

In [7]:
# Exercício 7: Redes e BFS
social_edges = [
    ("Idris", "Kamil"),
    ("Idris", "Talia"),
    ("Kamil", "Lina"),
    ("Lina", "Sasha"),
    ("Sasha", "Marco"),
    ("Marco", "Ken"),
    ("Ken", "Talia")
]

# Inicializa lista de adjacência
social_graph = {}
for u, v in social_edges:
    if u not in social_graph:
        social_graph[u] = []
    if v not in social_graph:
        social_graph[v] = []
    social_graph[u].append(v)
    social_graph[v].append(u)

def shortest_path(graph, src, dst):
    if src not in graph or dst not in graph:
        return None, 0
    if src == dst:
        return [src], 0

    queue = [[src]]
    visitados = {src}

    while queue:
        caminho = queue.pop(0)
        u = caminho[-1]
        if u == dst:
            return caminho, len(caminho) - 1

        # Explora adjacências em ordem lexicográfica
        for vizinho in sorted(graph[u]):
            if vizinho not in visitados:
                visitados.add(vizinho)
                queue.append(caminho + [vizinho])
    return None, 0

def path_with_mandatory(graph, src, dst, mandatory):
    caminho_total = [src]
    pendentes = set(mandatory)
    atual = src
    dist_total = 0

    while pendentes:
        # Busca obrigatório mais próximo
        proximo_no = None
        melhor_caminho = None
        menor_dist = float('inf')

        for p in sorted(list(pendentes)):
            c, d = shortest_path(graph, atual, p)
            if c and d < menor_dist:
                menor_dist = d
                proximo_no = p
                melhor_caminho = c

        if not proximo_no:
            # Sem caminho até obrigatórios
            return None, 0

        caminho_total.extend(melhor_caminho[1:])
        dist_total += menor_dist
        pendentes.remove(proximo_no)
        atual = proximo_no

    c_final, d_final = shortest_path(graph, atual, dst)
    if not c_final:
        return None, 0
    caminho_total.extend(c_final[1:])
    dist_total += d_final
    return caminho_total, dist_total

# Exercício 7: Testes
c, d = shortest_path(social_graph, "Idris", "Lina")
print(f"Caminho 'Idris' -> 'Lina': {c} (Distância: {d})")

c_m, d_m = path_with_mandatory(social_graph, "Idris", "Ken", {"Kamil", "Sasha"})
print(f"Caminho com obrigatórios (Kamil, Sasha) até Ken: {c_m} (Distância: {d_m})")

Caminho 'Idris' -> 'Lina': ['Idris', 'Kamil', 'Lina'] (Distância: 2)
Caminho com obrigatórios (Kamil, Sasha) até Ken: ['Idris', 'Kamil', 'Lina', 'Sasha', 'Marco', 'Ken'] (Distância: 5)


## Exercício 8: DFS e BFS como Geradores: Trilhas, Caminhos e Controle de Visitados

### Detalhes Teóricos e Impessoais
- **DFS como Gerador (`depth_first`)**: O gerador realiza busca recursiva produzindo `(vértice, caminho_acumulado)` e visitando cada vértice uma única vez. A pilha armazena o caminho percorrido para depuração rápida.
- **BFS como Gerador (`breadth_first`)**: O gerador utiliza uma fila de caminhos e retorna `(vértice, caminho_ótimo)`, garantindo que o primeiro caminho encontrado para cada nó seja de tamanho mínimo.
- **Mudança Consciente de Estruturas**: Utiliza-se um dicionário de adjacências cujas listas de vizinhos são mantidas ordenadas. Isso permite buscar adjacentes em tempo $O(1)$ e garante a ordem lexicográfica estável de varredura de forma nativa e sem custo de reordenação nas consultas.
- **Justificativa de Complexidade**:
  - **Tempo**: $O(V + E)$ para percorrer todo o grafo. A manutenção ordenada dos vizinhos custa $O(\text{grau} \log(\text{grau}))$ por vértice apenas no momento do cadastro.
  - **Memória**: $O(V + E)$ para a estrutura e $O(V)$ de espaço auxiliar de pilha/fila e conjunto de visitados.

In [8]:
# Exercício 8: Geradores de busca
def depth_first(graph, start):
    visitados = set()

    def dfs_helper(u, path):
        visitados.add(u)
        yield u, path
        for vizinho in sorted(graph[u]):
            if vizinho not in visitados:
                yield from dfs_helper(vizinho, path + [vizinho])

    yield from dfs_helper(start, [start])

def breadth_first(graph, start):
    visitados = {start}
    queue = [(start, [start])]
    while queue:
        u, path = queue.pop(0)
        yield u, path
        for vizinho in sorted(graph[u]):
            if vizinho not in visitados:
                visitados.add(vizinho)
                queue.append((vizinho, path + [vizinho]))

# Exercício 8: Testes
print("Trilha DFS a partir de Idris:")
for node, path in depth_first(social_graph, "Idris"):
    print(f"  Nó: {node}, Caminho: {path}")

print("Trilha BFS a partir de Idris:")
for node, path in breadth_first(social_graph, "Idris"):
    print(f"  Nó: {node}, Caminho: {path}")

Trilha DFS a partir de Idris:
  Nó: Idris, Caminho: ['Idris']
  Nó: Kamil, Caminho: ['Idris', 'Kamil']
  Nó: Lina, Caminho: ['Idris', 'Kamil', 'Lina']
  Nó: Sasha, Caminho: ['Idris', 'Kamil', 'Lina', 'Sasha']
  Nó: Marco, Caminho: ['Idris', 'Kamil', 'Lina', 'Sasha', 'Marco']
  Nó: Ken, Caminho: ['Idris', 'Kamil', 'Lina', 'Sasha', 'Marco', 'Ken']
  Nó: Talia, Caminho: ['Idris', 'Kamil', 'Lina', 'Sasha', 'Marco', 'Ken', 'Talia']
Trilha BFS a partir de Idris:
  Nó: Idris, Caminho: ['Idris']
  Nó: Kamil, Caminho: ['Idris', 'Kamil']
  Nó: Talia, Caminho: ['Idris', 'Talia']
  Nó: Lina, Caminho: ['Idris', 'Kamil', 'Lina']
  Nó: Ken, Caminho: ['Idris', 'Talia', 'Ken']
  Nó: Sasha, Caminho: ['Idris', 'Kamil', 'Lina', 'Sasha']
  Nó: Marco, Caminho: ['Idris', 'Talia', 'Ken', 'Marco']


## Exercício 9: Caminhos Globais e Conectividade em Rede Portuária

### Detalhes Teóricos e Impessoais
- **Floyd-Warshall**: Algoritmo clássico de programação dinâmica para calcular os menores caminhos entre todos os pares de vértices em $O(V^3)$ tempo.
- **Validação Cruzada**: A linha da matriz do porto correspondente a `Berco_A` é comparada com a execução de Dijkstra a partir de `Berco_A` para comprovar a exatidão.
- **Threshold de Pesos e Conectividade**: Ao desconsiderar arestas com peso maior que o limite (Threshold), simula-se a indisponibilidade de caminhos mais caros ou congestionados. O impacto é verificado executando um algoritmo de componentes conexas nas arestas filtradas.
- **Justificativa de Complexidade**:
  - **Tempo (Floyd-Warshall)**: $O(V^3)$, onde $V$ é a quantidade de vértices, pois utiliza três loops aninhados independentemente do número de arestas.
  - **Espaço (Floyd-Warshall)**: $O(V^2)$ para a matriz de distâncias.
  - **Impacto no Threshold**: Com threshold de 4, a rede se fragmenta em múltiplos componentes isolados. Conforme o threshold sobe para 6, a conectividade é restabelecida na maior parte das áreas, mostrando a dependência de vias de transporte de maior custo operacional para manter a fluidez global.

In [9]:
# Exercício 9: Rede portuária
porto_vertices = ["Berco_A", "Berco_B", "Patio_1", "Patio_2", "Patio_3", "Alfandega", "Centro_Logistico"]

porto_arestas = [
    ("Berco_A", "Patio_1", 4),
    ("Berco_A", "Patio_2", 7),
    ("Berco_B", "Patio_2", 3),
    ("Berco_B", "Patio_3", 6),
    ("Patio_1", "Patio_2", 2),
    ("Patio_2", "Patio_3", 2),
    ("Patio_1", "Alfandega", 8),
    ("Patio_2", "Alfandega", 5),
    ("Patio_3", "Centro_Logistico", 4),
    ("Alfandega", "Centro_Logistico", 3)
]

def floyd_warshall(vertices, edges):
    n = len(vertices)
    v_map = {v: i for i, v in enumerate(vertices)}
    # Inicializa matriz com infinito
    dist = [[float('inf')] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0

    for u, v, w in edges:
        ui, vi = v_map[u], v_map[v]
        dist[ui][vi] = w
        dist[vi][ui] = w

    # Loops de programação dinâmica
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]

    return dist, v_map

# Executa Floyd-Warshall
matriz_dist, v_map = floyd_warshall(porto_vertices, porto_arestas)

def dijkstra_verificar(vertices, edges, start):
    adj = {v: [] for v in vertices}
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))

    dist = {v: float('inf') for v in vertices}
    dist[start] = 0
    nao_visitados = set(vertices)

    while nao_visitados:
        u = min(nao_visitados, key=lambda n: dist[n])
        nao_visitados.remove(u)
        for vizinho, peso in adj[u]:
            if dist[u] + peso < dist[vizinho]:
                dist[vizinho] = dist[u] + peso
    return dist

# Validação cruzada
dijkstra_dist = dijkstra_verificar(porto_vertices, porto_arestas, "Berco_A")

print("Validação Dijkstra vs Floyd-Warshall a partir de Berco_A:")
idx_berco_a = v_map["Berco_A"]
for v in porto_vertices:
    idx_v = v_map[v]
    fw_val = matriz_dist[idx_berco_a][idx_v]
    dk_val = dijkstra_dist[v]
    print(f"  Para {v:18}: Floyd-Warshall = {fw_val:2} | Dijkstra = {dk_val:2} | Status: {'OK' if fw_val == dk_val else 'ERRO'}")

# Componentes conectados com Threshold
def componentes_threshold(vertices, edges, threshold):
    adj = {v: [] for v in vertices}
    for u, v, w in edges:
        if w <= threshold:
            adj[u].append(v)
            adj[v].append(u)

    visitados = set()
    componentes = []

    def dfs(u, comp):
        visitados.add(u)
        comp.append(u)
        for vizinho in adj[u]:
            if vizinho not in visitados:
                dfs(vizinho, comp)

    for v in vertices:
        if v not in visitados:
            comp = []
            dfs(v, comp)
            componentes.append(comp)
    return componentes

print(f"\nComponentes com Threshold = 4: {componentes_threshold(porto_vertices, porto_arestas, 4)}")
print(f"Componentes com Threshold = 6: {componentes_threshold(porto_vertices, porto_arestas, 6)}")

Validação Dijkstra vs Floyd-Warshall a partir de Berco_A:
  Para Berco_A           : Floyd-Warshall =  0 | Dijkstra =  0 | Status: OK
  Para Berco_B           : Floyd-Warshall =  9 | Dijkstra =  9 | Status: OK
  Para Patio_1           : Floyd-Warshall =  4 | Dijkstra =  4 | Status: OK
  Para Patio_2           : Floyd-Warshall =  6 | Dijkstra =  6 | Status: OK
  Para Patio_3           : Floyd-Warshall =  8 | Dijkstra =  8 | Status: OK
  Para Alfandega         : Floyd-Warshall = 11 | Dijkstra = 11 | Status: OK
  Para Centro_Logistico  : Floyd-Warshall = 12 | Dijkstra = 12 | Status: OK

Componentes com Threshold = 4: [['Berco_A', 'Patio_1', 'Patio_2', 'Berco_B', 'Patio_3', 'Centro_Logistico', 'Alfandega']]
Componentes com Threshold = 6: [['Berco_A', 'Patio_1', 'Patio_2', 'Berco_B', 'Patio_3', 'Centro_Logistico', 'Alfandega']]


## Exercício 10: Dijkstra: Menor Caminho por Custo e Análise de Eficiência

### Detalhes Teóricos e Impessoais
- **Dijkstra**: Algoritmo para encontrar os caminhos mínimos a partir de uma origem em grafos ponderados positivos.
- **Fila de Prioridade vs Varredura Linear**:
  - Com `use_heap=True`, o algoritmo extrai o nó de menor custo temporário em tempo $O(\log V)$ utilizando uma min-heap.
  - Com `use_heap=False`, o algoritmo percorre linearmente o conjunto de nós não visitados em tempo $O(V)$ para encontrar o menor valor.
- **Justificativa de Custo e Instrumentação**:
  - **Tempo (Heap)**: $O((V + E) \log V)$. Ideal para grafos esparsos ($E \ll V^2$).
  - **Tempo (Linear)**: $O(V^2 + E)$. Ideal para grafos densos ($E \approx V^2$).
  - **Instrumentação**: Registra-se o número de extrações de nós para demonstrar empiricamente a eficiência comparativa das duas estratégias.

In [10]:
# Exercício 10: Dijkstra completo
class WeightedGraph:
    def __init__(self, directed=False):
        self.adj = {}
        self.directed = directed

    def add_edge(self, u, v, w):
        if u not in self.adj:
            self.adj[u] = []
        if v not in self.adj:
            self.adj[v] = []
        self.adj[u].append((v, w))
        if not self.directed:
            self.adj[v].append((u, w))

def dijkstra_com_escolha(graph, src, use_heap=True):
    dist = {v: float('inf') for v in graph.adj}
    dist[src] = 0
    prev = {v: None for v in graph.adj}
    extracao_contador = 0

    if use_heap:
        # Heap binária instrumentada
        from collections import namedtuple
        HeapItem = namedtuple('HeapItem', ['d', 'v'])

        # Implementa ordem determinística por valor
        class MinHeapItem:
            def __init__(self, d, v):
                self.d = d
                self.v = v
            def __lt__(self, other):
                if self.d == other.d:
                    return self.v < other.v
                return self.d < other.d

        heap = BinaryHeap()
        heap.push(MinHeapItem(0, src))
        visitados = set()

        while len(heap) > 0:
            item = heap.pop()
            u = item.v
            if u in visitados:
                continue
            visitados.add(u)
            extracao_contador += 1

            if u not in graph.adj:
                continue
            for vizinho, peso in graph.adj[u]:
                if dist[u] + peso < dist[vizinho]:
                    dist[vizinho] = dist[u] + peso
                    prev[vizinho] = u
                    heap.push(MinHeapItem(dist[vizinho], vizinho))
    else:
        nao_visitados = set(graph.adj.keys())
        while nao_visitados:
            u = min(nao_visitados, key=lambda node: dist[node])
            nao_visitados.remove(u)
            extracao_contador += 1

            if dist[u] == float('inf'):
                continue
            if u not in graph.adj:
                continue
            for vizinho, peso in graph.adj[u]:
                if dist[u] + peso < dist[vizinho]:
                    dist[vizinho] = dist[u] + peso
                    prev[vizinho] = u

    return dist, prev, extracao_contador

# Exercício 10: Testes
g_porto = WeightedGraph(directed=False)
for u, v, w in porto_arestas:
    g_porto.add_edge(u, v, w)

d_h, _, count_h = dijkstra_com_escolha(g_porto, "Berco_A", use_heap=True)
d_l, _, count_l = dijkstra_com_escolha(g_porto, "Berco_A", use_heap=False)

print("Custo das rotas de Berco_A:")
for k, v in d_h.items():
    print(f"  {k:18}: {v}")
print(f"\nUso de Heap: {count_h} extrações")
print(f"Uso Linear: {count_l} extrações")

Custo das rotas de Berco_A:
  Berco_A           : 0
  Patio_1           : 4
  Patio_2           : 6
  Berco_B           : 9
  Patio_3           : 8
  Alfandega         : 11
  Centro_Logistico  : 12

Uso de Heap: 7 extrações
Uso Linear: 7 extrações


## Exercício 11: Árvore Geradora Mínima: Prim com Desempate Determinístico

### Detalhes Teóricos e Impessoais
- **Algoritmo de Prim**: Constrói a árvore geradora mínima (MST) a partir de um vértice de início, expandindo a árvore de forma gulosa ao adicionar a aresta de menor peso que conecta a árvore a um nó externo.
- **Desempate Determinístico**: As arestas candidatas são priorizadas e ordenadas pela tupla `(peso, u, v)` em ordem lexicográfica, garantindo que o resultado final seja estável e idêntico em qualquer execução.
- **Justificativa de Custo**:
  - **Tempo**: $O(V^2)$ na implementação com busca linear de arestas, ou $O(E \log V)$ usando fila de prioridades.
  - **MST vs Caminho Mínimo**: A MST minimiza a soma total das arestas para conectar todos os nós acessíveis, enquanto o menor caminho de Dijkstra minimiza a distância individual de um único nó de origem para os demais.

In [11]:
# Exercício 11: Prim determinístico
def prim(graph, start):
    visitados = {start}
    mst_edges = []
    total_cost = 0
    n = len(graph.adj)

    while len(visitados) < n:
        candidatos = []
        for u in visitados:
            if u not in graph.adj:
                continue
            for v, w in graph.adj[u]:
                if v not in visitados:
                    # Registra aresta candidata
                    candidatos.append((w, u, v))

        if not candidatos:
            # Grafo desonexo detectado
            print("LOG: Componente alcançável concluído.")
            break

        # Desempate determinístico (lexicográfico)
        candidatos.sort(key=lambda item: (item[0], item[1], item[2]))
        w, u, v = candidatos[0]
        visitados.add(v)
        mst_edges.append((u, v, w))
        total_cost += w

    return mst_edges, total_cost

# Grafo com 8 vértices
g_teste = WeightedGraph(directed=False)
arestas_mst_teste = [
    ("A", "B", 2), ("A", "C", 3), ("B", "C", 1), ("B", "D", 4),
    ("C", "E", 5), ("D", "E", 1), ("D", "F", 7), ("E", "G", 8),
    ("F", "G", 2), ("F", "H", 3), ("G", "H", 6)
]
for u, v, w in arestas_mst_teste:
    g_teste.add_edge(u, v, w)

arestas_mst, custo_mst = prim(g_teste, "A")
print("Arestas selecionadas para a MST:")
for u, v, w in arestas_mst:
    print(f"  {u} -- {v} com peso {w}")
print(f"Custo total da MST: {custo_mst}")

Arestas selecionadas para a MST:
  A -- B com peso 2
  B -- C com peso 1
  B -- D com peso 4
  D -- E com peso 1
  D -- F com peso 7
  F -- G com peso 2
  F -- H com peso 3
Custo total da MST: 20


## Exercício 12: Exercício Integrador: Logística Urbana com Janelas de Tempo

### Detalhes Teóricos e Impessoais
- **NP-Dificuldade (TSP-TW)**: O problema de encontrar a rota ótima visitando múltiplos locais com janelas de tempo é conhecido como TSP com Janelas de Tempo (TSP-TW). Ele é NP-difícil porque estende o Problema do Caixeiro Viajante (TSP), no qual o número de permutações viáveis cresce fatorialmente $O(V!)$.
- **Heurística Gulosa com Heap**: A cada passo, calcula-se o score de cada entrega pendente a partir da posição atual no instante $t$ e seleciona-se a de melhor pontuação. A pontuação é computada pela fórmula:
  $$\text{score} = \text{prioridade} \times 10 - \text{custo\_deslocamento} - \text{penalidade\_atraso}$$
  - A `penalidade_atraso` é aplicada se a chegada ocorrer após `janela_fim`, sendo proporcional ao atraso em minutos.
  - Se a chegada for antes de `janela_inicio`, o tempo de espera é adicionado ao tempo de viagem.
- **Escolha do Hub Inicial**: Seleciona-se o **Centro** como hub de partida. Esta decisão baseia-se na centralidade geográfica de suas conexões diretas (`Centro -> Botafogo`, `Centro -> Tijuca`, `Centro -> Madureira`) e na proximidade de janelas de tempo iniciais como Copacabana (janela 10-45) e Tijuca (janela 15-60), o que minimiza a ociosidade inicial.
- **Escolha do Algoritmo de Menores Caminhos**: Executam-se múltiplas chamadas do algoritmo de Dijkstra a partir dos pontos de interesse. Como o número de pontos de interesse é pequeno ($P=8$), o custo de múltiplas chamadas é de $O(P \times (V \log V + E))$, o que é consideravelmente inferior ao custo do Floyd-Warshall ($O(V^3)$) no grafo inteiro.

In [12]:
# Exercício 12: Roteamento logístico
bairros_edges = [
    ("Centro", "Botafogo", 18),
    ("Centro", "Tijuca", 16),
    ("Centro", "Madureira", 34),
    ("Botafogo", "Copacabana", 10),
    ("Botafogo", "Ipanema", 14),
    ("Botafogo", "Centro", 20),
    ("Copacabana", "Ipanema", 9),
    ("Copacabana", "Botafogo", 12),
    ("Copacabana", "Centro", 28),
    ("Ipanema", "Copacabana", 10),
    ("Ipanema", "Botafogo", 16),
    ("Ipanema", "Barra", 30),
    ("Tijuca", "Centro", 18),
    ("Tijuca", "Madureira", 26),
    ("Tijuca", "Botafogo", 22),
    ("Madureira", "Tijuca", 30),
    ("Madureira", "Centro", 35),
    ("Madureira", "Jacarepagua", 28),
    ("Jacarepagua", "Barra", 18),
    ("Jacarepagua", "Madureira", 26),
    ("Barra", "Jacarepagua", 16),
    ("Barra", "Ipanema", 32),
    ("Barra", "Centro", 40)
]

g_rio = WeightedGraph(directed=True)
for u, v, w in bairros_edges:
    g_rio.add_edge(u, v, w)

# Consultas de adjacência
print("Vizinhos de Centro:", g_rio.adj.get("Centro", []))
print("Vizinhos de Madureira:", g_rio.adj.get("Madureira", []))

def dijkstra_rio(graph, src):
    dist = {v: float('inf') for v in graph.adj}
    dist[src] = 0
    prev = {v: None for v in graph.adj}
    nao_visitados = set(graph.adj.keys())

    while nao_visitados:
        u = min(nao_visitados, key=lambda node: dist[node])
        nao_visitados.remove(u)
        if dist[u] == float('inf'):
            continue
        for vizinho, peso in graph.adj.get(u, []):
            if dist[u] + peso < dist[vizinho]:
                dist[vizinho] = dist[u] + peso
                prev[vizinho] = u
    return dist, prev

# Calcula caminhos de bairros
custo_caminhos = {}
predecessores = {}
for b in g_rio.adj.keys():
    d, p = dijkstra_rio(g_rio, b)
    custo_caminhos[b] = d
    predecessores[b] = p

def travel_cost(u, v):
    # Retorna custo do caminho
    c = custo_caminhos.get(u, {}).get(v, float('inf'))
    if c == float('inf'):
        return float('inf'), []
    # Reconstrói caminho
    caminho = []
    curr = v
    pais = predecessores[u]
    while curr is not None:
        caminho.append(curr)
        curr = pais[curr]
    caminho.reverse()
    return c, caminho

deliveries = [
    # bairro, janela_ini, janela_fim, prioridade
    ("Copacabana", 10, 45, 4),
    ("Ipanema", 25, 75, 5),
    ("Tijuca", 15, 60, 3),
    ("Madureira", 60, 130, 3),
    ("Jacarepagua", 80, 150, 2),
    ("Botafogo", 20, 70, 2)
]

def roteamento_entregas(start_hub):
    t = 0  # minutos desde 09:00
    pos_atual = start_hub
    entregas_restantes = list(deliveries)
    rota_bairros = [start_hub]
    atrasos = []
    logs = []

    class ScoredDelivery:
        def __init__(self, score, index, delivery, cost, wait, arr_time):
            self.score = score
            self.index = index
            self.delivery = delivery
            self.cost = cost
            self.wait = wait
            self.arr_time = arr_time
        def __lt__(self, other):
            # Desempate determinístico da heap
            if self.score == other.score:
                return self.delivery[0] < other.delivery[0]
            return self.score > other.score

    while entregas_restantes:
        heap_pontos = BinaryHeap()
        # Avalia cada entrega pendente
        for idx, ent in enumerate(entregas_restantes):
            bairro, j_ini, j_fim, prio = ent
            cost, _ = travel_cost(pos_atual, bairro)

            if cost == float('inf'):
                continue

            arr_time = t + cost
            wait = max(0, j_ini - arr_time)
            total_arrival = arr_time + wait

            # Penalidade se atrasar
            penalidade = 0
            if total_arrival > j_fim:
                penalidade = (total_arrival - j_fim) * 2

            # Score de priorização
            score = prio * 10 - cost - wait - penalidade
            heap_pontos.push(ScoredDelivery(score, idx, ent, cost, wait, total_arrival))

        if len(heap_pontos) == 0:
            # Bairros inalcançáveis restantes
            break

        escolha = heap_pontos.pop()
        bairro, j_ini, j_fim, prio = escolha.delivery
        entregas_restantes.pop(escolha.index)

        # Avança tempo
        t_antes = t
        t = escolha.arr_time
        status = "No prazo"
        if t > j_fim:
            status = "Atrasado"
            atrasos.append((bairro, t, t - j_fim))

        pos_atual = bairro
        rota_bairros.append(bairro)
        logs.append(f"Escolha: {bairro} | t: {t_antes}->{t} | custo: {escolha.cost} | espera: {escolha.wait} | status: {status}")

    # Retorna para o melhor hub
    custo_centro, path_centro = travel_cost(pos_atual, "Centro")
    custo_barra, path_barra = travel_cost(pos_atual, "Barra")

    if custo_centro < custo_barra:
        rota_bairros.extend(path_centro[1:])
        t += custo_centro
    else:
        rota_bairros.extend(path_barra[1:])
        t += custo_barra

    return rota_bairros, t, atrasos, logs

# Executa roteamento
rota, tempo_total, atrasos, logs = roteamento_entregas("Centro")
print("\n--- LOG DE EXECUÇÃO ---")
for line in logs:
    print(line)
print("\n--- RELATÓRIO FINAL ---")
print("Rota completa:", " -> ".join(rota))
print(f"Tempo total do turno: {tempo_total} minutos")
print("Entregas fora da janela:")
for b, arr, atr in atrasos:
    print(f"  {b}: chegou às {int(9 + arr//60):02d}:{int(arr%60):02d} (Atraso: {atr} min)")

Vizinhos de Centro: [('Botafogo', 18), ('Tijuca', 16), ('Madureira', 34)]
Vizinhos de Madureira: [('Tijuca', 30), ('Centro', 35), ('Jacarepagua', 28)]

--- LOG DE EXECUÇÃO ---
Escolha: Ipanema | t: 0->32 | custo: 32 | espera: 0 | status: No prazo
Escolha: Copacabana | t: 32->42 | custo: 10 | espera: 0 | status: No prazo
Escolha: Botafogo | t: 42->54 | custo: 12 | espera: 0 | status: No prazo
Escolha: Madureira | t: 54->108 | custo: 54 | espera: 0 | status: No prazo
Escolha: Jacarepagua | t: 108->136 | custo: 28 | espera: 0 | status: No prazo
Escolha: Tijuca | t: 136->192 | custo: 56 | espera: 0 | status: Atrasado

--- RELATÓRIO FINAL ---
Rota completa: Centro -> Ipanema -> Copacabana -> Botafogo -> Madureira -> Jacarepagua -> Tijuca -> Centro
Tempo total do turno: 210 minutos
Entregas fora da janela:
  Tijuca: chegou às 12:12 (Atraso: 132 min)
